# Part 3 - Machine Learning: predict no-shows and choose the model honestly
**Project: CareLoop Clinics - No-Show Guard**

**Goal:** a risk score for every booking that the front desk can use **the evening before the
appointment** to decide whom to phone. At that moment everything we use as a feature is already
known: the patient's age, distance, fee, how long ago they booked, how many times they missed
before, and whether today's reminder was sent.

Only about 1 booking in 5 is a no-show, so accuracy alone is misleading. We judge every model
by **recall**, **precision** and **ROC-AUC** as well.

**The rule for this whole notebook:** the test set is for *reporting only*. We never use it to
choose a model, a setting or a threshold. All choices are made on the training set with
cross-validation.

| Step | What we do |
|---|---|
| 3.1 | Build the features, split 80/20, compute the baseline |
| 3.2 | Logistic regression (with scaling) and what its coefficients mean |
| 3.3 | Decision tree: unlimited depth vs depth 3 (overfitting) |
| 3.4 | Random forest and a comparison table |
| 3.5 | Tune, choose the model and the threshold with cross-validation, then test once |

In [1]:
# Install the libraries this notebook needs (safe to re-run).
%pip install -q --disable-pip-version-check pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

# On Google Colab the project files are not there yet, so we download (clone) the repo first.
if "google.colab" in str(get_ipython()) and not Path("MASAI-CAPSTONE-PROJ").exists():
    os.system("git clone -q https://github.com/Advay4706/MASAI-CAPSTONE-PROJ")
if Path("MASAI-CAPSTONE-PROJ").exists():
    os.chdir("MASAI-CAPSTONE-PROJ")

BASE = Path(".") if Path("data").exists() else Path("..")   # the project's top folder
OUT = BASE / "outputs"

RANDOM_STATE = 42     # the same "seed" everywhere, so every run gives the same result
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

scikit-learn 1.8.0 | pandas 3.0.1 | numpy 2.4.3


## Step 3.1 - Features, split and baseline

**What:** turn the clean table into a feature table `X` (numbers only) and a target `y`
(`no_show`, 1 or 0). Split the rows 80% for training and 20% for testing. Work out the
*baseline*: the accuracy of a "model" that always says "Attended".

**Why each choice:**

- **Features used:** `age`, `distance_km`, `fee_inr`, `lead_days`, `prior_no_shows`, `reminder_sent`,
  `booking_channel`, `visit_type`.
- **Not used:** `appointment_id` and `patient_id` (labels, not information), the dates (already
  summarised by `lead_days`), `clinic_id` and `gender` (left out by the brief), and `status`
  (it *is* the answer - using it would be cheating, called **target leakage**).
- **`.map({"Yes": 1, "No": 0})`** turns a two-value text column into 1/0.
- **One-hot encoding** (`pd.get_dummies`) turns a text column with several values into one
  1/0 column per value. We cannot write App=1, Phone=2, Walk-In=3, because that would tell the
  model that Walk-In is "three times" App, which means nothing.
- **Train/test split:** the model learns from the training rows and is graded on test rows it
  has never seen. `random_state=42` makes the random split repeatable.
- We do **not** re-sort the rows: the split depends on the row order of the file.

In [3]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(OUT / "appointments_clean.csv")        # keep the file's row order
print("Loaded:", df.shape)

feature_columns = ["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
                   "reminder_sent", "booking_channel", "visit_type"]

X = df[feature_columns].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})          # text -> 1/0
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])          # one-hot encoding
y = df["no_show"]                                                         # the target

print("X has", X.shape[1], "columns:")
print(list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print("\nTraining rows:", len(X_train), "| test rows:", len(X_test))
print("No-shows in the test set:", int(y_test.sum()), f"({y_test.mean():.1%})")
print("No-shows in the training set:", int(y_train.sum()), f"({y_train.mean():.1%})")

# Baseline: always predict "Attended" (0). It is right for every patient who attended.
baseline_accuracy = (y_test == 0).mean()
print(f"\nBaseline accuracy (always predict 'Attended'): {baseline_accuracy:.4f} "
      f"= {(y_test == 0).sum()} of {len(y_test)}")

Loaded: (990, 16)
X has 11 columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Training rows: 792 | test rows: 198
No-shows in the test set: 49 (24.7%)
No-shows in the training set: 169 (21.3%)

Baseline accuracy (always predict 'Attended'): 0.7525 = 149 of 198


**Reading the result**

- `X` has **11 columns**: 6 numeric ones, 3 for `booking_channel` and 2 for `visit_type`.
- **792** training rows and **198** test rows; **49** of the test rows are no-shows.
- **Baseline accuracy is 0.7525** (149 of 198). A model that never flags anyone is 75% "accurate"
  and completely useless - it catches zero no-shows. This is why accuracy alone is a trap on
  **imbalanced** data, and why a real model must beat 0.7525 *and* have useful recall.

## Step 3.2 - Logistic regression

**What:** scale the features, fit a logistic regression, and report its test metrics and
coefficients.

**The idea:** logistic regression adds up `coefficient x feature` for all features and squeezes
the total through an S-shaped curve to get a probability between 0 and 1. A positive
coefficient pushes the risk up, a negative one pushes it down.

**Why scale:** `fee_inr` is in the hundreds, `reminder_sent` is 0 or 1. `StandardScaler` puts
every feature on the same footing (mean 0, standard deviation 1), which helps the fitting and
makes the coefficients comparable in size.

**Fit the scaler on the training set only.** The test set stands for future, unseen data. If
its mean and standard deviation leaked into the scaler, the test would no longer be honest
(**data leakage**). So: `fit_transform` on train, only `transform` on test.

**The metrics** (a "positive" is a no-show):

| Metric | Question it answers |
|---|---|
| Accuracy | Of all predictions, how many were right? |
| Precision | Of the patients we flagged, how many really missed? |
| Recall | Of the patients who really missed, how many did we flag? |
| F1 | One number that balances precision and recall |
| ROC-AUC | If we pick one no-show and one attender at random, how often does the no-show get the higher risk score? 0.5 = coin flip, 1.0 = perfect. It does not depend on a threshold. |

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)     # learn mean/std from TRAIN, then scale train
X_test_s = scaler.transform(X_test)           # scale test with the TRAIN mean/std

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_s, y_train)

lr_pred = log_reg.predict(X_test_s)                    # 0/1 using the default 0.5 threshold
lr_prob = log_reg.predict_proba(X_test_s)[:, 1]        # probability of class 1 (no-show)

results = {}      # we collect every model's test metrics here for the comparison table
results["Logistic regression"] = {
    "accuracy": accuracy_score(y_test, lr_pred),
    "recall": recall_score(y_test, lr_pred),
    "roc_auc": roc_auc_score(y_test, lr_prob),
}

print(f"Accuracy : {accuracy_score(y_test, lr_pred):.3f}   (baseline {baseline_accuracy:.4f})")
print(f"Precision: {precision_score(y_test, lr_pred):.3f}")
print(f"Recall   : {recall_score(y_test, lr_pred):.3f}")
print(f"F1       : {f1_score(y_test, lr_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, lr_prob):.3f}")

cm = confusion_matrix(y_test, lr_pred)
print("\nConfusion matrix (rows = actual, columns = predicted):")
display(pd.DataFrame(cm, index=["actual Attended", "actual No-Show"],
                     columns=["predicted Attended", "predicted No-Show"]))

Accuracy : 0.788   (baseline 0.7525)
Precision: 0.630
Recall   : 0.347
F1       : 0.447
ROC-AUC  : 0.789

Confusion matrix (rows = actual, columns = predicted):


,predicted Attended,predicted No-Show
actual Attended,139,10
actual No-Show,32,17


### Coefficients
One coefficient per feature, in the column order of `X`. Because the features were scaled,
each coefficient is the effect of moving that feature by **one standard deviation**.

In [5]:
coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": log_reg.coef_[0],
    "feature_std (scaler.scale_)": scaler.scale_,
})
display(coef_table.round(3))

i = list(X.columns).index("reminder_sent")
coef_reminder = log_reg.coef_[0][i]
std_reminder = scaler.scale_[i]

odds_per_std = np.exp(coef_reminder)                    # effect of +1 standard deviation
odds_no_to_yes = np.exp(coef_reminder / std_reminder)   # effect of the real change No (0) -> Yes (1)

print(f"reminder_sent coefficient (scaled feature): {coef_reminder:.3f}")
print(f"standard deviation of reminder_sent        : {std_reminder:.3f}")
print(f"np.exp(coef)         = {odds_per_std:.3f}  -> odds x {odds_per_std:.2f} per ONE STANDARD DEVIATION")
print(f"np.exp(coef / scale) = {odds_no_to_yes:.3f}  -> odds x {odds_no_to_yes:.2f} for the real change No -> Yes "
      f"({1 - odds_no_to_yes:.0%} lower odds)")

,feature,coefficient,feature_std (scaler.scale_)
0,age,-0.254,16.656
1,distance_km,0.253,6.052
2,fee_inr,0.453,184.254
3,lead_days,0.614,8.614
4,prior_no_shows,0.650,0.969
5,reminder_sent,-0.581,0.495
6,booking_channel_App,-0.047,0.486
7,booking_channel_Phone,0.216,0.481
8,booking_channel_Walk-In,-0.186,0.435
9,visit_type_Consultation,0.016,0.493


reminder_sent coefficient (scaled feature): -0.581
standard deviation of reminder_sent        : 0.495
np.exp(coef)         = 0.560  -> odds x 0.56 per ONE STANDARD DEVIATION
np.exp(coef / scale) = 0.309  -> odds x 0.31 for the real change No -> Yes (69% lower odds)


**Reading the result**

- Accuracy **0.788** beats the 0.7525 baseline, and ROC-AUC is **0.789** - the model ranks
  patients by risk clearly better than chance.
- But at the default 0.5 threshold, **recall is only 0.347**: it catches 17 of the 49 no-shows
  and misses 32 (confusion matrix `[[139, 10], [32, 17]]`). Precision is 0.630 (17 of the 27
  flagged really missed), F1 is 0.447. We fix the low recall with a better threshold in Step 3.5.
- `reminder_sent` has the most negative coefficient (**-0.581**). `prior_no_shows` (+0.650)
  and `lead_days` (+0.614) are the strongest positive ones - the same lead-time story that
  the SQL bands showed in Part 2.

**The two numbers for `reminder_sent`, and why they differ**

- `np.exp(coef)` = **0.56**. This is the change in odds for a one-*standard-deviation* step in
  the scaled feature. A reminder is not "one standard deviation"; its standard deviation is 0.495.
- `np.exp(coef / scale)` = **0.31**. Dividing by the standard deviation converts the coefficient
  back to the feature's real units, so this is the change in odds for the real step from
  No (0) to Yes (1).
- **They differ because the model was fitted on scaled features: going from No to Yes is about
  two standard deviations (1 / 0.495), not one.** Reporting 0.56 ("44% lower") would understate it.

**Conclusion, worded carefully:** in this model, holding the other features constant, appointments
with a reminder have about **69% lower odds** of a no-show than appointments without one. This is
an **association, not proof of cause**: the data are observational. Reminders were not handed out
at random - maybe app users, who are more organised anyway, get more reminders. Only an experiment
(send reminders to a random half) could prove that reminders *cause* fewer no-shows.

(Odds are not probability: odds = p / (1 - p). "69% lower odds" does not mean "69% fewer no-shows".)

## Step 3.3 - Decision tree and overfitting

**What:** fit a decision tree twice - once with no depth limit, once limited to 3 levels - and
compare training accuracy with test accuracy.

**The idea:** a decision tree asks a chain of yes/no questions ("lead_days > 14?",
"reminder sent?") and ends in a leaf that gives the prediction. Trees need no scaling, because
a question like "fee > 700?" does not care about units.

**Overfitting:** with no depth limit a tree keeps splitting until every training row is
classified correctly. It has then *memorised* the training data, including its random noise, and
does badly on new data. The warning sign is a **big gap between training and test accuracy**.
`max_depth=3` forces the tree to keep only the few strongest patterns.

In [6]:
from sklearn.tree import DecisionTreeClassifier

tree_full = DecisionTreeClassifier(random_state=RANDOM_STATE)                 # no depth limit
tree_full.fit(X_train, y_train)

tree_3 = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)       # at most 3 questions deep
tree_3.fit(X_train, y_train)

rows = []
for name, tree in [("Unlimited depth", tree_full), ("max_depth = 3", tree_3)]:
    train_acc = tree.score(X_train, y_train)      # .score() = accuracy
    test_acc = tree.score(X_test, y_test)
    rows.append({"tree": name, "actual depth": tree.get_depth(), "leaves": tree.get_n_leaves(),
                 "train accuracy": train_acc, "test accuracy": test_acc, "gap": train_acc - test_acc})
display(pd.DataFrame(rows).round(3))
print(f"Baseline accuracy to beat: {baseline_accuracy:.4f}")

tree3_pred = tree_3.predict(X_test)
results["Decision tree (depth 3)"] = {
    "accuracy": accuracy_score(y_test, tree3_pred),
    "recall": recall_score(y_test, tree3_pred),
    "roc_auc": roc_auc_score(y_test, tree_3.predict_proba(X_test)[:, 1]),
}

,tree,actual depth,leaves,train accuracy,test accuracy,gap
0,Unlimited depth,15,164,1.000,0.672,0.328
1,max_depth = 3,3,8,0.822,0.778,0.044


Baseline accuracy to beat: 0.7525


**Reading the result**

- **Unlimited tree:** training accuracy **1.00**, test accuracy **0.672**, depth 15. The gap of
  0.33 is textbook overfitting. Its test accuracy is even *below the 0.7525 baseline* - it
  memorised noise and is worse than predicting "Attended" for everyone.
- **Depth-3 tree:** training **0.822**, test **0.778**. The gap is small (0.04), so what it
  learned carries over to new patients.
- **In one sentence:** a large gap between training and test accuracy shows the model has
  memorised the training data instead of learning patterns that generalise; limiting the depth
  trades a little training accuracy for much better behaviour on unseen data.

## Step 3.4 - Random forest and a comparison table

**What:** fit a random forest of 100 trees, report its metrics and its three most important
features, then put the three models side by side.

**The idea:** one deep tree overfits. A random forest grows many deep trees, each on a random
sample of the rows and with a random subset of features at each split, and lets them **vote**.
Individual trees make different mistakes, and averaging cancels much of that out.

**Feature importance** (`rf.feature_importances_`): how much each feature helped the trees make
purer splits, averaged over the forest. The values add up to 1. It tells us *how much* a feature
is used, not the *direction* of its effect.

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]          # probability of a no-show, needed for AUC

results["Random forest (100 trees)"] = {
    "accuracy": accuracy_score(y_test, rf_pred),
    "recall": recall_score(y_test, rf_pred),
    "roc_auc": roc_auc_score(y_test, rf_prob),
}

print(f"Training accuracy: {rf.score(X_train, y_train):.3f}")
print(f"Test accuracy    : {accuracy_score(y_test, rf_pred):.3f}   (baseline {baseline_accuracy:.4f})")
print(f"Test recall      : {recall_score(y_test, rf_pred):.3f}")
print(f"Test ROC-AUC     : {roc_auc_score(y_test, rf_prob):.3f}")

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 3 feature importances:")
print(importances.head(3).round(3).to_string())
print(f"\n(all 11 importances add up to {importances.sum():.2f})")

Training accuracy: 1.000
Test accuracy    : 0.747   (baseline 0.7525)
Test recall      : 0.306
Test ROC-AUC     : 0.706

Top 3 feature importances:
lead_days      0.220
age            0.197
distance_km    0.195

(all 11 importances add up to 1.00)


### Comparison table (test set, for reporting)

In [8]:
comparison = pd.DataFrame(results).T[["accuracy", "recall", "roc_auc"]]
display(comparison.round(3))
print("Highest test ROC-AUC:", comparison["roc_auc"].idxmax())

,accuracy,recall,roc_auc
Logistic regression,0.788,0.347,0.789
Decision tree (depth 3),0.778,0.224,0.700
Random forest (100 trees),0.747,0.306,0.706


Highest test ROC-AUC: Logistic regression


**Reading the result**

- The forest also reaches training accuracy 1.00, but its test accuracy is **0.747**, recall
  **0.306** and ROC-AUC **0.706**. Voting makes it better than the single unlimited tree
  (0.672), but it still sits right at the baseline for accuracy.
- Top three importances: **`lead_days` (0.220), `age` (0.197), `distance_km` (0.195)**. Lead time is
  first again. (Features with many different values, like age and distance, tend to get high
  importance in forests because they offer many places to split - a known bias.)
- **Logistic regression looks best:** highest accuracy (0.788), highest recall (0.347) and
  clearly the highest ROC-AUC (0.789 vs 0.700 and 0.706). The likely reason is that with only
  792 training rows and effects that are mostly "more of X means more risk", a simple model
  that cannot overfit much does better than flexible ones.
- This table uses the test set, so it is **for reporting only**. The formal choice of model is
  made in Step 3.5 using the training data alone.

## Step 3.5 - Tune, choose the model and the threshold, then test once

**The problem:** if we pick the model that scores best on the test set, the test set has helped
to make a decision, and its score is no longer an honest estimate for new data.

**The solution: cross-validation (CV) on the training set.** With `cv=5` the 792 training rows
are cut into 5 parts ("folds"). The model is trained on 4 parts and scored on the 5th, five
times, so that every row is used for validation exactly once. The average of the 5 scores is
the CV score. The test set stays locked away.

Plan:

- (a) `GridSearchCV`: try several settings of the random forest, each scored by 5-fold CV.
- (b) Compare the CV ROC-AUC of all candidate models. The highest wins.
- (c) Choose the decision threshold using **out-of-fold** probabilities.
- (d) Fit the winner on all training rows and evaluate it **once** on the test set.
- (e) Save the risk scores.

### (a) Tune the random forest with `GridSearchCV`
A **hyperparameter** is a setting we choose before training (number of trees, maximum depth).
`GridSearchCV` tries every combination in the grid (2 x 3 = 6 here), runs 5-fold CV for each
(30 fits), and remembers the best one. `best_score_` is a cross-validated ROC-AUC - not accuracy.

In [9]:
from sklearn.model_selection import GridSearchCV, cross_val_predict, cross_val_score

param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}

grid = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), param_grid, cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)            # training set only

print("best_params_:", grid.best_params_)
print(f"best_score_ (cross-validated ROC-AUC): {grid.best_score_:.4f}")

grid_table = pd.DataFrame(grid.cv_results_)[["param_max_depth", "param_n_estimators", "mean_test_score"]]
grid_table = grid_table.rename(columns={"mean_test_score": "cv_roc_auc"}).sort_values("cv_roc_auc", ascending=False)
display(grid_table.round(4))

best_params_: {'max_depth': 3, 'n_estimators': 200}
best_score_ (cross-validated ROC-AUC): 0.7673


,param_max_depth,param_n_estimators,cv_roc_auc
1,3,200,0.7673
0,3,100,0.7658
3,5,200,0.7656
2,5,100,0.7631
5,8,200,0.7621
4,8,100,0.7575


### (b) Choose the model by cross-validated ROC-AUC (training set only)
For logistic regression we use a **pipeline**: scaler + model chained into one object. Inside
each CV fold the pipeline re-fits the scaler on that fold's own training part. If we scaled the
whole training set first, every validation fold would have influenced the scaler - a small leak.
So the pipeline gets the **unscaled** `X_train`.

In [10]:
from sklearn.pipeline import Pipeline

lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

candidates = {
    "Logistic regression (pipeline)": lr_pipe,
    "Decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
    "Random forest (100 trees)": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

cv_auc = {}
for name, model in candidates.items():
    cv_auc[name] = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc").mean()

# The tuned forest was already scored by the same 5-fold CV inside GridSearchCV.
candidates["Random forest (tuned)"] = RandomForestClassifier(random_state=RANDOM_STATE, **grid.best_params_)
cv_auc["Random forest (tuned)"] = grid.best_score_

cv_table = pd.Series(cv_auc, name="cv_roc_auc (train)").sort_values(ascending=False)
display(cv_table.round(4).to_frame())

winner_name = cv_table.idxmax()
winner = candidates[winner_name]
print("Winner (highest cross-validated ROC-AUC):", winner_name)

,cv_roc_auc (train)
Logistic regression (pipeline),0.7737
Random forest (tuned),0.7673
Random forest (100 trees),0.7466
Decision tree (depth 3),0.6784


Winner (highest cross-validated ROC-AUC): Logistic regression (pipeline)


### (c) Choose the threshold with out-of-fold probabilities
The model gives a probability. The **threshold** turns it into a decision: flag the booking if
`risk >= threshold`. The default 0.5 is not a law - on imbalanced data it flags too few.

`cross_val_predict` gives every training row a probability from a model that **did not see that
row** during training (an *out-of-fold* prediction). So we can compare thresholds honestly
without touching the test set. We pick the threshold with the higher F1.

In [11]:
oof_prob = cross_val_predict(winner, X_train, y_train, cv=5, method="predict_proba")[:, 1]

threshold_rows = []
for threshold in [0.5, 0.35]:
    flagged = (oof_prob >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_train, flagged),
        "recall": recall_score(y_train, flagged),
        "f1": f1_score(y_train, flagged),
        "flagged": int(flagged.sum()),
    })
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.round(3))

best_row = threshold_table.loc[threshold_table["f1"].idxmax()]
THRESHOLD = float(best_row["threshold"])
print(f"Chosen threshold (higher F1): {THRESHOLD}")
print(f"Out of {len(y_train)} training rows, {int(y_train.sum())} are real no-shows.")

,threshold,precision,recall,f1,flagged
0,0.50,0.575,0.249,0.347,73
1,0.35,0.513,0.467,0.489,154


Chosen threshold (higher F1): 0.35
Out of 792 training rows, 169 are real no-shows.


**Why recall matters here:** the two kinds of mistake do not cost the same. A *missed warning*
(a no-show we did not flag) costs an empty doctor slot. An *extra call* (a patient we phoned who
would have come anyway) costs about a minute of front-desk time. So it is worth accepting lower
precision to catch many more no-shows.

### (d) Fit the winner on the training set and evaluate ONCE on the test set

In [12]:
winner.fit(X_train, y_train)                         # the pipeline fits its own scaler on the training rows

test_prob = winner.predict_proba(X_test)[:, 1]       # risk score for each test booking
test_flag = (test_prob >= THRESHOLD).astype(int)     # 1 = phone this patient

final_cm = confusion_matrix(y_test, test_flag)
print("Final model:", winner_name, "| threshold:", THRESHOLD)
print(f"Precision: {precision_score(y_test, test_flag):.3f}")
print(f"Recall   : {recall_score(y_test, test_flag):.3f}")
print(f"Flagged  : {int(test_flag.sum())} of {len(test_flag)} test bookings")
print(f"ROC-AUC  : {roc_auc_score(y_test, test_prob):.3f}")
display(pd.DataFrame(final_cm, index=["actual Attended", "actual No-Show"],
                     columns=["not flagged", "flagged"]))

# For information only (NOT used for any choice): the tuned forest on the test set.
tuned_test_auc = roc_auc_score(y_test, grid.best_estimator_.predict_proba(X_test)[:, 1])
print(f"For information only - tuned random forest test ROC-AUC: {tuned_test_auc:.3f}")

Final model: Logistic regression (pipeline) | threshold: 0.35
Precision: 0.571
Recall   : 0.571
Flagged  : 49 of 198 test bookings
ROC-AUC  : 0.789


,not flagged,flagged
actual Attended,128,21
actual No-Show,21,28


For information only - tuned random forest test ROC-AUC: 0.741


### (e) Save the risk scores
`X_test.index` holds the original row numbers, so we can look up each test row's
`appointment_id` in the loaded table.

In [13]:
risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": test_prob.round(6),
    "flag": test_flag,
})
risk_scores.to_csv(OUT / "test_risk_scores.csv", index=False)

saved = pd.read_csv(OUT / "test_risk_scores.csv")
print("Saved rows:", len(saved), "| columns:", list(saved.columns))
print("Rows with flag = 1:", int(saved["flag"].sum()), "(reported above:", int(test_flag.sum()), ")")
print("flag agrees with risk_prob >= threshold:", bool(((saved["risk_prob"] >= THRESHOLD).astype(int) == saved["flag"]).all()))
display(saved.sort_values("risk_prob", ascending=False).head(5))

Saved rows: 198 | columns: ['appointment_id', 'no_show', 'risk_prob', 'flag']
Rows with flag = 1: 49 (reported above: 49 )
flag agrees with risk_prob >= threshold: True


,appointment_id,no_show,risk_prob,flag
68,A0635,1,0.937204,1
24,A0544,1,0.924066,1
141,A0293,1,0.919634,1
106,A0950,1,0.818616,1
114,A0996,0,0.766710,1


**Reading the result**

- **(a)** The best forest setting is `max_depth=3, n_estimators=200`, with a cross-validated
  ROC-AUC of **0.767**. Shallow trees win - the same lesson as Step 3.3.
- **(b)** Cross-validated ROC-AUC on the training set: **logistic regression 0.774**, tuned forest
  0.767, default forest 0.747, depth-3 tree 0.678. **Logistic regression wins**, and it is also
  the easiest model to explain to a clinic manager.
- **(c)** Out-of-fold results on the 792 training rows (169 real no-shows):

  | Threshold | Precision | Recall | F1 | Flagged |
  |---|---|---|---|---|
  | 0.50 | 0.575 | 0.249 | 0.347 | 73 |
  | 0.35 | 0.513 | 0.467 | 0.489 | 154 |

  **0.35 is chosen**: F1 is higher and recall almost doubles (25% -> 47%) for a small drop in precision.
- **(d)** One test-set evaluation of logistic regression at 0.35: **precision 0.571, recall 0.571,
  49 of 198 flagged, ROC-AUC 0.789.** The front desk would phone 49 patients and reach 28 of the
  49 real no-shows; 21 calls go to patients who would have come anyway. Compare with Step 3.2 at
  threshold 0.5: only 17 of 49 caught.
- **(e)** `outputs/test_risk_scores.csv` has 198 rows; the 49 rows with `flag = 1` match the
  number reported.
- The tuned forest's test ROC-AUC (0.741) is shown for information only; it played no part in the choice.

**How a clinic would really choose the threshold:** by capacity and cost - if the front desk can
make about 50 calls in an evening, set the threshold so that about 50 bookings are flagged, and
weigh that calling time against the cost of each empty slot.

**Honest limits of this model** (stated again in the README):

1. The medians and outlier fences in Part 1 were computed on all 990 rows before the split, so
   the test rows had a small indirect influence on the cleaning.
2. The same patient can have several bookings, so a random split can put one patient in both
   train and test.
3. The split is random; a real system is trained on the past and used on the future.

## Part 3 summary

| Question | Answer |
|---|---|
| Baseline to beat | Accuracy 0.7525 (always "Attended"), which catches nobody |
| Model chosen | Logistic regression (scaler + model pipeline), CV ROC-AUC 0.774 |
| Threshold chosen | 0.35 (F1 0.489 vs 0.347 at 0.5, on out-of-fold predictions) |
| Final test result | ROC-AUC 0.789, precision 0.571, recall 0.571, 49 of 198 flagged |
| Strongest signals | prior no-shows and long lead time raise risk; a sent reminder goes with lower risk |